# 04 Phase Retrieval

            Loads the HDF5 data dictionary, runs the unified recipe-driven phase
            retrieval, plots the CDI results, and writes results and recipe back
            into the same HDF5 file.

In [ ]:
import os, sys
from os.path import join
from getpass import getuser
from importlib import reload

import h5py
import numpy as np
import matplotlib.pyplot as plt
import skimage.morphology
from pyFAI.detectors import Detector


def find_basefolder(start=None):
    return os.path.abspath(start or os.getcwd())


BASEFOLDER = find_basefolder()
sys.path.append(join(BASEFOLDER, "library"))
print("Base folder:", BASEFOLDER)
import fthcore as fth
import helper_functions as helper
import interactive
from interactive import cimshow
import mask_lib
import reconstruct_rb as rec
import PETRA_MaxP04_loading as loading
import fth_phase_workflow as wf

wf = reload(wf)  # Refresh helpers when rerunning in an existing kernel.

try:
    import cupy as cp
    import cupyx as cpx
    import CCI_core_cupy as cci
    import Phase_Retrieval as PhR

    GPU = True
    print("GPU available")
except Exception:
    import CCI_core as cci

    PhR = None
    GPU = False
    print("GPU unavailable")

%matplotlib widget
try:
    %load_ext jupyter_black
except Exception:
    pass

In [ ]:
import phase_retrieval_core_unified as pr

In [ ]:
def selected_mode(recon, mode=0):
    modes = wf.as_modes(recon)
    mode = min(int(mode), modes.shape[0] - 1)
    return modes[mode]

## Load data

In [ ]:
BASEFOLDER = find_basefolder()
USER = "rb"
im_id = 415  # Positive-helicity image ID produced by 01_FTH.ipynb.
topo_id = 416  # Reference/negative-helicity image ID in the same HDF5 file.
# True passes the centered mask_pixel saved by 01_FTH to phase retrieval.
# Masked detector pixels are excluded from the image constraint.
USE_MASK_PIXEL = True
folder_general = helper.create_folder(join(BASEFOLDER, "processed"))
folder_logs = helper.create_folder(join(folder_general, "Logs"))
DATA_H5 = join(folder_logs, f"data_recon_ImId_{im_id:04d}_{USER}.hdf5")

data = wf.load_data_dict(DATA_H5)
experimental_setup = data["experimental_setup"]
positive_label = data["positive_label"]
reference_label = data["reference_label"]
loaded_im_ids = data["holo"][positive_label]["id"]
loaded_topo_ids = data["holo"][reference_label]["id"]
loaded_im_id = int(
    loaded_im_ids[0]
    if isinstance(loaded_im_ids, (list, tuple, np.ndarray))
    else loaded_im_ids
)
loaded_topo_id = int(
    loaded_topo_ids[0]
    if isinstance(loaded_topo_ids, (list, tuple, np.ndarray))
    else loaded_topo_ids
)
if loaded_im_id != im_id:
    raise ValueError(
        f"Requested im_id={im_id}, but the HDF5 contains im_id={loaded_im_id}."
    )
if loaded_topo_id != topo_id:
    raise ValueError(
        f"Requested topo_id={topo_id}, but the HDF5 contains topo_id={loaded_topo_id}."
    )
labels = wf.get_hologram_labels(data)
print("Phase retrieval labels:", labels)
pol1, pol2 = "+", "-"
phase_retrieval_labels = [label for label in (pol1, pol2) if label in labels]
phase_retrieval_labels += [
    label for label in labels if label not in phase_retrieval_labels
]
if pol1 not in labels or pol2 not in labels:
    raise ValueError(
        f'The FTH_CDI_01 recipe expects "{pol1}" and "{pol2}" in data["holo"]. '
        f"Available labels are {labels}."
    )

if "supportmask" not in data:
    raise ValueError("Run 03_define_supportmask.ipynb before phase retrieval.")

supportmask = np.asarray(data["supportmask"])
# data['mask_pixel'] was already centered in 01_FTH; do not center it twice.
stored_mask_pixel = np.asarray(
    data.get("mask_pixel", np.zeros_like(supportmask)), dtype=np.uint8
)
if stored_mask_pixel.shape != supportmask.shape:
    raise ValueError(
        f"mask_pixel shape {stored_mask_pixel.shape} != supportmask shape {supportmask.shape}"
    )
mask_pixel = (
    stored_mask_pixel
    if USE_MASK_PIXEL
    else np.zeros_like(stored_mask_pixel, dtype=np.uint8)
)
print(f"mask_pixel filtering: {'enabled' if USE_MASK_PIXEL else 'disabled'}")
focus_fth = dict(data.get("focus_fth", data.get("focus", {})))
focus_cdi = dict(data.get("focus_cdi", {}))
if "roi" in focus_fth:
    roi_fth = np.asarray(focus_fth["roi"], dtype=int)
else:
    roi_fth = np.array([0, supportmask.shape[0], 0, supportmask.shape[1]])
print("FTH ROI:", roi_fth)
holograms = {
    label: np.asarray(data["holo"][label]["image_c"], dtype=float)
    for label in phase_retrieval_labels
}

## Recipe

In [ ]:
# Exact formulation from FTH_CDI_01.ipynb.
offset_vmin = 2
Startimage = None
Startgamma = None

phase_retrieval_holograms = {
    label: holograms[label] for label in phase_retrieval_labels
}
primary_label = phase_retrieval_labels[0]
secondary_labels = phase_retrieval_labels[1:]
full_labels = [primary_label, primary_label, *secondary_labels]
partial_labels = [primary_label, primary_label, *secondary_labels]
recipe_labels = full_labels + partial_labels

times = 1
phase_retrieval_recipe = {
    "algorithm_list": ["HAPRE", "ER", "ER"] * times,
    "number_iterations": [500, 50, 50] * times,
    "helicity": ["+", "+", "-"] * times,
    "beta_zero": 0.5,
    "beta_mode": ["arctan", "const", "const"] * times,
    "alpha_zero": 0.0,
    "alpha_mode": "const",
    "RL_its": [0, 0, 0, 50, 50, 50][: 3 * times],
    "RL_freqs": [1e9, 1e9, 1e9, 20, 20, 20][: 3 * times],
    "TV_freqs": 1e9,
    "plot_every": 50,
    "average_img": 10,
    "Fourier_last": True,
    "Startimage": [None, "+", "+", "+", "+", "+"][: 3 * times],
    "Startgamma": [None, None, None, None, "+", "+"][: 3 * times],
    "hologram_intensity_cutoff_vmin": offset_vmin,
    "hologram_offset": 0.0,  # Avoid double subtraction; offset_vmin handles the baseline.
    "output": [False, True, True, False, True, True][: 3 * times],
    # Legacy semantics: [1, N] uses the original support for mode 1 and
    # a spatially N-enlarged support for mode 2.
    "modes": [1, 2],
    "normalize_startimage_between_holograms": True,
    "subtract_startimage_fit_intercept": False,  # Set True for legacy initialization.
    "return_format": "auto",
    "crop": 350,
}
phase_retrieval_recipe
# Standard NumPy names used by the other reconstruction notebooks.
pos = np.asarray(phase_retrieval_holograms[positive_label], dtype=float)
neg = np.asarray(phase_retrieval_holograms[reference_label], dtype=float)


In [ ]:
def get_supportmask_coordinates(sample):
    """
    Dictionary that stores coordinates of circular support mask apertures.
    Taken from FTH_CDI_01.ipynb.
    """
    off = 0
    support_coord = dict()
    support_coord["kri2"] = [
        (1024 + off, 1024 + off, 14.0),
        (687.50 + off, 620 + off, 83.0),
    ]
    return support_coord[sample]


# Select stored coordinates, or replace this with your own list of
# (y, x, radius) tuples. Skip this cell to retain the stored supportmask.
sample = "kri2"
support_coordinates = get_supportmask_coordinates(sample)
supportmask = mask_lib.create_circle_supportmask(
    support_coordinates, phase_retrieval_holograms["+"].shape
)
supportmask = (supportmask > 0).astype(np.uint8)

In [ ]:
# Diagnostic: show the exact intensities and exclusion masks sent to retrieval.
_offset_spec = phase_retrieval_recipe["hologram_offset"]
_cutoff_vmin = phase_retrieval_recipe["hologram_intensity_cutoff_vmin"]
prepared_holograms = {}
prepared_bsmasks = {}
fig, axes = plt.subplots(
    len(phase_retrieval_labels),
    2,
    figsize=(10, 4 * len(phase_retrieval_labels)),
    squeeze=False,
)
for row, label in enumerate(phase_retrieval_labels):
    _offset = float(
        _offset_spec.get(label, 0.0) if isinstance(_offset_spec, dict) else _offset_spec
    )
    _input = np.asarray(phase_retrieval_holograms[label], dtype=float) - _offset
    if _cutoff_vmin >= 0:
        _finite_nonzero = _input[(_input != 0) & np.isfinite(_input)]
        if _finite_nonzero.size:
            _input = _input - np.nanpercentile(_finite_nonzero, _cutoff_vmin)
    _input = np.where(np.isnan(_input), 0, _input)
    _bsmask = np.asarray(mask_pixel).copy()
    _bsmask[_input < 0] = 1
    _input = np.clip(_input, 0, None)
    prepared_holograms[label] = _input
    prepared_bsmasks[label] = _bsmask
    _vmin, _vmax = wf.finite_percentile_limits(_input, (1, 99.9))
    axes[row, 0].imshow(_input, vmin=_vmin, vmax=_vmax, cmap="viridis")
    axes[row, 0].set_title(f"{label}: phase-retrieval input intensity")
    axes[row, 1].imshow(_bsmask > 0, vmin=0, vmax=1, cmap="gray")
    axes[row, 1].set_title(f"{label}: bsmask used (white = excluded)")
    axes[row, 0].set_axis_off()
    axes[row, 1].set_axis_off()
plt.tight_layout()
plt.show()

## Run phase retrieval

In [ ]:
phase_retrieval_result = pr.phase_retrieval_algorithm(
    phase_retrieval_holograms,
    mask_pixel,
    supportmask,
    phase_retrieval_recipe,
)

retrieved_holograms = phase_retrieval_result["full_coherence"]
retrieved_holograms_pc = phase_retrieval_result["partial_coherence"]
retrieved_holograms_gradient = phase_retrieval_result["gradient_descent"]
bsmasks = phase_retrieval_result["bsmasks"]
gammas = phase_retrieval_result["gamma"]
errors = phase_retrieval_result["error"]
print("Phase retrieval done.")

for label in phase_retrieval_labels:
    data["holo"][label]["retrieved_full"] = retrieved_holograms.get(label)
    data["holo"][label]["retrieved_pc"] = retrieved_holograms_pc.get(label)
    data["holo"][label]["retrieved_gradient"] = retrieved_holograms_gradient.get(label)
    data["holo"][label]["bsmask"] = bsmasks.get(label)
    data["holo"][label]["gamma"] = gammas.get(label)

error_summary = {
    "steps": [
        {
            "step": step["step"],
            "helicity": step["helicity"],
            "mode": step["mode"],
            "Nit": step["Nit"],
            "RL_it": step["RL_it"],
            "RL_freq": step["RL_freq"],
            "coherence": step["coherence"],
            "output": step["output"],
            "error": np.asarray(step["error"]),
            "support_error": np.asarray(step["support_error"]),
        }
        for step in errors["steps"]
    ],
}

In [ ]:
# Detector-plane diagnostic: retrieved intensity and excluded-pixel mask.
fig, axes = plt.subplots(
    len(phase_retrieval_labels),
    2,
    figsize=(10, 4 * len(phase_retrieval_labels)),
    squeeze=False,
)
for row, label in enumerate(phase_retrieval_labels):
    field = (
        retrieved_holograms_pc.get(label)
        if retrieved_holograms_pc.get(label) is not None
        else retrieved_holograms.get(label)
    )
    if field is None:
        axes[row, 0].text(0.5, 0.5, "No retrieved field", ha="center", va="center")
        axes[row, 0].set_axis_off()
    else:
        reconstructed_hologram = np.sum(np.abs(wf.as_modes(field)) ** 2, axis=0)
        vmin, vmax = wf.finite_percentile_limits(reconstructed_hologram, (1, 99.9))
        axes[row, 0].imshow(reconstructed_hologram, vmin=vmin, vmax=vmax, cmap="viridis")
        axes[row, 0].set_title(f"{label}: retrieved hologram intensity")
        axes[row, 0].set_axis_off()
    binary_bsmask = np.asarray(bsmasks[label]) > 0
    axes[row, 1].imshow(binary_bsmask, vmin=0, vmax=1, cmap="gray")
    axes[row, 1].set_title(f"{label}: bsmask (white = excluded)")
    axes[row, 1].set_axis_off()
plt.tight_layout()
plt.show()

## CDI reconstruction and focus

In [ ]:
pol1 = "+"
pol2 = "-"
retrieved_type = (
    "retrieved_full"  # options: "retrieved_full", "retrieved_pc", "retrieved_gradient"
)

for pol in (pol1, pol2):
    available = [
        key
        for key in ("retrieved_full", "retrieved_pc", "retrieved_gradient")
        if data["holo"][pol].get(key) is not None
    ]
    print(pol, "available:", available)

pos_retrieved = data["holo"][pol1][retrieved_type]
neg_retrieved = data["holo"][pol2][retrieved_type]
if pos_retrieved is None or neg_retrieved is None:
    raise ValueError(f"{retrieved_type} is not available for {pol1} and/or {pol2}.")
pos_retrieved = np.asarray(pos_retrieved)
neg_retrieved = np.asarray(neg_retrieved)

phase_cdi = focus_cdi.get("phase", 0)
prop_dist_cdi = focus_cdi.get("prop_dist", 0)
dx = focus_cdi.get("dx", 0)
dy = focus_cdi.get("dy", 0)
focus_mode_cdi = int(focus_cdi.get("mode", 0))
use_bs = False
bs_diam_cdi = 25

retrieved_shape = wf.spatial_shape(pos_retrieved)
if use_bs:
    mask_bs_cdi = 1 - cci.circle_mask(
        retrieved_shape, np.array(retrieved_shape) / 2, bs_diam_cdi, sigma=4
    )
else:
    mask_bs_cdi = np.ones(retrieved_shape)

crop = int(phase_retrieval_recipe["crop"])
crop_shape = np.array(supportmask.shape) - 2 * crop
if tuple(crop_shape) != retrieved_shape:
    raise ValueError(
        f"Expected retrieved shape {tuple(crop_shape)}, got {retrieved_shape}."
    )

roi_crop = wf.rescale_roi(roi_fth, supportmask.shape, crop_shape)
roi_crop_s = wf.roi_to_slices(roi_crop)
roi_crop_shape = (roi_crop[1] - roi_crop[0], roi_crop[3] - roi_crop[2])
print("roi_fth:", roi_fth)
print("roi_crop:", roi_crop, "shape:", roi_crop_shape, "of", tuple(crop_shape))

p_cdi_all = wf.reconstruct_cdi_modes(
    pos_retrieved,
    mask_bs_cdi,
    fth,
    experimental_setup,
    prop_dist=prop_dist_cdi,
    phase=phase_cdi,
    dx=dx,
    dy=dy,
)
n_cdi_all = wf.reconstruct_cdi_modes(
    neg_retrieved,
    mask_bs_cdi,
    fth,
    experimental_setup,
    prop_dist=prop_dist_cdi,
    phase=phase_cdi,
    dx=dx,
    dy=dy,
)
p_cdi = selected_mode(p_cdi_all, focus_mode_cdi)
n_cdi = selected_mode(n_cdi_all, focus_mode_cdi)
print("phase_cdi:", phase_cdi)
print("prop_dist_cdi:", prop_dist_cdi)
print("focus_mode_cdi:", focus_mode_cdi)

In [ ]:
# Optional fine tuning. Use the sliders, then execute the following cell to store values.
mode = "+"
supportmask_eff = wf.resize_binary_to_shape(supportmask, wf.spatial_shape(p_cdi))
focus_sliders = rec.focusCDI(
    pos_retrieved * mask_bs_cdi,
    neg_retrieved * mask_bs_cdi,
    roi_crop_s,
    mask=supportmask_eff,
    phase=phase_cdi,
    dx=dx,
    dy=dy,
    prop_dist=prop_dist_cdi,
    experimental_setup=experimental_setup,
    operation=mode,
    max_prop_dist=3,
    scale=(2, 98),
)
slider_prop, slider_phase, slider_dx, slider_dy = focus_sliders[:4]
slider_mode = focus_sliders[4] if len(focus_sliders) > 4 else None

In [ ]:
phase_cdi = slider_phase.value
prop_dist_cdi = slider_prop.value
dx = slider_dx.value
dy = slider_dy.value
focus_mode_cdi = int(slider_mode.value) if slider_mode is not None else 0

p_cdi_all = wf.reconstruct_cdi_modes(
    pos_retrieved,
    mask_bs_cdi,
    fth,
    experimental_setup,
    prop_dist=prop_dist_cdi,
    phase=phase_cdi,
    dx=dx,
    dy=dy,
)
n_cdi_all = wf.reconstruct_cdi_modes(
    neg_retrieved,
    mask_bs_cdi,
    fth,
    experimental_setup,
    prop_dist=prop_dist_cdi,
    phase=phase_cdi,
    dx=dx,
    dy=dy,
)
p_cdi = selected_mode(p_cdi_all, focus_mode_cdi)
n_cdi = selected_mode(n_cdi_all, focus_mode_cdi)
print("Updated phase_cdi:", phase_cdi)
print("Updated prop_dist_cdi:", prop_dist_cdi)
print("Updated focus_mode_cdi:", focus_mode_cdi)

## Plot and save

In [ ]:
recon_cdi_full = np.log(p_cdi) - np.log(n_cdi)
cdi_shape = wf.spatial_shape(recon_cdi_full)
supportmask_eff = wf.resize_binary_to_shape(supportmask, cdi_shape)
recon_cdi_roi = wf.spatial_roi(recon_cdi_full, roi_crop_s)
supportmask_roi = supportmask_eff[roi_crop_s]
recon_cdi = wf.apply_spatial_mask(recon_cdi_roi, supportmask_roi)
print("Plotting CDI ROI:", roi_crop, "shape:", recon_cdi.shape)

fig, ax = plt.subplots(figsize=(5, 5))
tmp = np.real(recon_cdi)
vmin, vmax = wf.finite_percentile_limits(tmp)
ax.imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax.set_title(f"{pol1} - {pol2} {retrieved_type}, mode {focus_mode_cdi}")
ax.set_axis_off()

png_name = join(folder_general, f"PhR_recon_ImId_{int(im_id):04d}_{USER}.png")
plt.savefig(png_name, bbox_inches="tight", transparent=False, dpi=200)
plt.show()
print("Saved figure:", png_name)
recon = np.asarray(recon_cdi)


In [ ]:
png_name = join(folder_general, f"PhR_recon_ImId_{int(im_id):04d}_{USER}.png")

# Always write the display PNG in the same final cell as the HDF5 result.
fig, ax = plt.subplots(figsize=(5, 5))
tmp = np.real(recon_cdi)
vmin, vmax = wf.finite_percentile_limits(tmp)
ax.imshow(tmp, vmin=vmin, vmax=vmax, cmap="gray")
ax.set_title(f"{pol1} - {pol2} {retrieved_type}, mode {focus_mode_cdi}")
ax.set_axis_off()
fig.savefig(png_name, bbox_inches="tight", transparent=False, dpi=200)
plt.close(fig)

focus_cdi = {
    "prop_dist": prop_dist_cdi,
    "phase": phase_cdi,
    "dx": dx,
    "dy": dy,
    "roi": roi_fth,
    "roi_crop": roi_crop,
    "mode": focus_mode_cdi,
    "operation": mode,
    "retrieved_type": retrieved_type,
    "pol1": pol1,
    "pol2": pol2,
}

for key in [
    "dark_id_im",
    "dark_id_topo",
    "im_id",
    "topo_id",
    "fth_hologram",
    "fth_hologram_unmasked",
    "fth_png_title",
    "fth_recon",
    "fth_recon_unmasked",
    "mask_pixel_smooth",
    "mask_multiplier",
    "sum_c",
    "diff_c",
    "mask_pixel_c",
    "mask_pixel_c_png",
    "prop_dist",
    "phase",
    "dx",
    "dy",
    "focus_operation",
    "roi",
    "recon_cdi",
    "recon_topo_cdi",
    "phase_retrieval_png",
    "roi_cdi",
    "retrieved_type",
    "phase_cdi",
    "prop_dist_cdi",
    "dx_cdi",
    "dy_cdi",
    "focus_mode_cdi",
    "roi_crop",
    "mask_bs_cdi",
]:
    data.pop(key, None)
data["phase_retrieval_recipe"] = phase_retrieval_recipe
data["phase_retrieval_errors"] = error_summary
data["focus_cdi"] = focus_cdi
data["recon_cdi"] = recon_cdi
data["phase_retrieval_png"] = png_name

wf.save_data_dict(data, DATA_H5, overwrite=True)
print("Saved HDF5:", DATA_H5)
print("Saved PNG:", png_name)

In [ ]:
# Workflow summary
_summary_data = data if "data" in globals() and isinstance(data, dict) else {}
_summary_h5 = globals().get("DATA_H5", _summary_data.get("data_file", "n/a"))
_summary_holo = _summary_data.get("holo", {})
_summary_pos = _summary_data.get(
    "positive_label", globals().get("positive_label", None)
)
_summary_ref = _summary_data.get(
    "reference_label", globals().get("reference_label", None)
)
_summary_im = _summary_holo.get(_summary_pos, {}).get(
    "id", globals().get("im_id", "n/a")
)
_summary_topo = _summary_holo.get(_summary_ref, {}).get(
    "id", globals().get("topo_id", "n/a")
)
print("im_ids:", _summary_im)
print("topo_ids:", _summary_topo)
print("dark_ids (+):", _summary_holo.get(_summary_pos, {}).get("dark_id"))
print("dark_ids (-):", _summary_holo.get(_summary_ref, {}).get("dark_id"))
print("HDF5:", _summary_h5)

In [ ]:
# Acquisition ID summary
_id_holo = data.get("holo", {})
print("im_ids:", {label: state.get("id") for label, state in _id_holo.items()})
print("dark_ids:", {label: state.get("dark_id") for label, state in _id_holo.items()})